In [2]:
# ⚡ Real-world analogy

# Think of it like cooking:

# Step 1: Check ingredients
# Step 2: If missing → buy
# Step 3: Then cook

# 👉 You can’t do it in one step
# 👉 You need state + decisions

# ✅ Control flow graph / agent workflow
# 🔥 Key terminology (important)

# | Concept        | Meaning                      |
# | -------------- | ---------------------------- |
# | Tool chaining  | Output → next tool input     |
# | State          | Storing intermediate values  |
# | Branching      | If/else logic                |
# | Planner        | Decides multi-step execution |
# | Workflow agent | Structured execution         |


In [17]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage  # Use langchain_core

# === 1) Define tools ===

@tool
def get_weather(location: str) -> str:
    """Return a simple weather report with a numeric temperature."""
    return f"The weather in {location} is 21°C."

@tool
def process_temperature(temp: int) -> str:
    """If above 25 → multiply by 2; else subtract 5."""
    if temp > 25:
        return f"{temp} > 25 → multiplied = {temp * 2}"
    else:
        return f"{temp} ≤ 25 → subtracted = {temp - 5}"

# === 2) Initialize LLM ===

llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="not-needed",
    model="local-model",
    temperature=0.2,
)

# === 3) Workflow executor ===

def run_conditional_workflow(location: str):
    # Step 1 — call weather tool
    # Call the tool function directly since it's decorated with @tool
    weather_report = get_weather.invoke({"location": location})  # Use invoke with dict
    print("Weather report tool output:", weather_report)

    # Step 2 — extract temperature using LLM
    extract_prompt = f"""
Extract the integer temperature from the following text.
Only output the number:

{weather_report}
"""
    # Pass a list of messages
    extracted = llm.invoke([
        SystemMessage(content="Extract numeric temperature from text"),
        HumanMessage(content=extract_prompt)
    ])

    # Extract the temperature from the response
    temp_str = extracted.content.strip()
    temp = int(temp_str)
    print("Extracted temperature:", temp)

    # Step 3 — call the conditional processing tool
    result = process_temperature.invoke({"temp": temp})  # Use invoke with dict
    print("Conditional processing result:", result)

    # Step 4 — return final combined result
    return f"{weather_report}\n{result}"



In [18]:
# === 4) Run workflow ===

final_answer = run_conditional_workflow("Bangalore")
print("\n=== FINAL ANSWER ===\n", final_answer)

Weather report tool output: The weather in Bangalore is 21°C.
Extracted temperature: 21
Conditional processing result: 21 ≤ 25 → subtracted = 16

=== FINAL ANSWER ===
 The weather in Bangalore is 21°C.
21 ≤ 25 → subtracted = 16
